Data Preparation and Feature Engineering

Vehicle Valuation and Depreciation Intelligence Platform

Objective

The objective of this notebook is to prepare the used-vehicle listing data
for a baseline multiple linear regression model that predicts current listing
price.

The preparation process applies findings from exploratory data analysis,
including duplicate records, redundant columns, missing values, implausible
mileage observations, and high-cardinality categorical features.




In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_PATH = Path("../data/processed/used_cars_reduced.csv")
df = pd.read_csv(DATA_PATH)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

C:\Users\casey\AppData\Local\Temp\ipykernel_3212\3415453124.py:5: DtypeWarning: Columns (0: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [9]:
pd.set_option("display.max_columns", None)


In [10]:
"index" in df.columns

False

In [11]:
df

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


In [12]:
df.shape


(2663251, 21)

PHASE 1 i
Do new vehicles belong in the dataset?
Yes New vehicles provide the starting market reference before mileage, age, ownership, accidents, and other factors reduce value.


CHECK POINT 1. Target Price Validation

In [13]:
df['price'].isna()

0          False
1          False
2          False
3          False
4          False
           ...  
2663246    False
2663247    False
2663248    False
2663249    False
2663250    False
Name: price, Length: 2663251, dtype: bool

In [14]:
df['price'].isna().sum()

np.int64(0)

In [15]:
(df["price"] == 0).sum()

np.int64(0)

In [16]:
(df["price"] < 0).sum()

np.int64(0)

In [17]:
df['price'].max()
df.loc[df["price"] == df["price"].max()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1261170,Palm Harbor,34683,V12,V12,False,Gasoline,False,660.0,False,Ferrari,5339.0,Enzo,2.0,3299995.0,False,4.4,A,2 Dr STD Coupe,NaN,RWD,2003


In [18]:
df['price'].min()
df.loc[df["price"] == df["price"].min()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,False,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999


In [19]:
df['mileage'] 

0              7.0
1              8.0
2              NaN
3             11.0
4              7.0
            ...   
2663246    41897.0
2663247        5.0
2663248    57992.0
2663249    27857.0
2663250    22600.0
Name: mileage, Length: 2663251, dtype: float64

In [20]:

comparison=df.loc[(df["make_name"]== 'Buick') & 
                  (df["model_name"] == "Century"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison.shape
comparison.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,1999,190000.0,4.0,False,False,False,165.0
878365,2005,202158.0,5.0,True,False,False,249.0
301631,2001,175689.0,2.0,False,False,False,495.0
1148231,2002,NaN,5.0,False,False,False,600.0
1209120,2002,150000.0,4.0,False,False,False,750.0
1687931,2003,160730.0,4.0,True,False,False,995.0
1148579,1999,130474.0,3.0,True,False,False,1400.0
1738903,2002,236760.0,4.0,True,False,False,1490.0
44862,2001,185524.0,2.0,False,False,False,1499.0
1523587,2004,165385.0,5.0,False,False,False,1500.0


In [21]:
df["frame_damaged"].any()

np.True_

In [22]:
df.sort_values("price").head(5)

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,False,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999
1359718,Medley,33178,NaN,NaN,False,NaN,False,210.0,False,Ford,150000.0,Explorer,4.0,200.0,False,4.000000,A,XLT V6,NaN,RWD,2005
878365,Traverse City,49684,V6,V6,False,Gasoline,True,175.0,False,Buick,202158.0,Century,5.0,249.0,False,3.593750,A,Custom Sedan FWD,NaN,FWD,2005
1271928,Bainbridge,39817,NaN,NaN,False,NaN,True,170.0,False,Nissan,NaN,Altima Coupe,5.0,250.0,False,1.000000,CVT,2.5 S,NaN,FWD,2008
1271923,Bainbridge,39817,V6,V6,False,Gasoline,False,290.0,False,Acura,NaN,RL,3.0,250.0,False,1.000000,A,SH-AWD with Navigation and Tech Package,NaN,AWD,2006


In [23]:
(df["price"] < 500).sum()

np.int64(86)

In [24]:
(df["price"] < 1000).sum()

np.int64(494)

In [26]:
df.loc[
    df["price"] < 1000,
    [
        "make_name",
        "model_name",
        "year",
        "is_new",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,is_new,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,Buick,Century,1999,False,190000.0,4.0,False,False,False,165.0
1359718,Ford,Explorer,2005,False,150000.0,4.0,False,False,False,200.0
878365,Buick,Century,2005,False,202158.0,5.0,True,False,False,249.0
1271941,Kia,Sorento,2006,False,NaN,6.0,True,False,False,250.0
1271923,Acura,RL,2006,False,NaN,3.0,False,False,False,250.0
1271928,Nissan,Altima Coupe,2008,False,NaN,5.0,True,False,False,250.0
1271918,Toyota,Avalon,2005,False,NaN,5.0,True,False,False,250.0
1271913,Mitsubishi,Lancer,2008,False,NaN,2.0,True,False,False,250.0
1938898,Mercury,Villager,1998,False,104000.0,3.0,False,False,False,256.0
1579657,Pontiac,Vibe,2003,False,200000.0,5.0,True,False,False,299.0


In [27]:
comparison_Nissan=df.loc[(df["make_name"]== 'Nissan') & 
                  (df["model_name"] == "Altima"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison_Nissan.shape
comparison_Nissan.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1394034,2020,NaN,1.0,False,False,False,386.9
1320684,2004,175222.0,7.0,True,False,False,484.0
1320691,1998,177420.0,2.0,False,False,False,484.0
949965,1997,NaN,3.0,True,False,False,550.0
1579680,2006,204952.0,4.0,True,False,False,677.0
1677446,2005,200829.0,6.0,True,False,False,800.0
1250485,2003,191652.0,NaN,False,False,False,899.0
950307,2002,NaN,4.0,False,False,False,900.0
559459,2003,NaN,2.0,True,False,False,900.0
950193,1997,NaN,4.0,True,False,False,900.0


In [28]:
df.loc[df["price"] < 1000, "price"].value_counts()

price
999.0    95
995.0    64
900.0    32
484.0    27
495.0    23
         ..
389.0     1
980.0     1
256.0     1
890.0     1
449.0     1
Name: count, Length: 72, dtype: int64

In [29]:
df.loc[
    df["price"] == 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
2686,Honda,Civic,1996,222434.0,Teterboro,False,4.0,True,False,False,7608,999.0
2690,Nissan,Maxima,2001,NaN,Teterboro,False,3.0,True,False,False,7608,999.0
126057,Nissan,Sentra,2005,205000.0,East Granby,False,2.0,False,False,False,6026,999.0
151782,Mercedes-Benz,420-Class,1987,203189.0,New Windsor,False,2.0,False,False,False,12553,999.0
158302,Ford,Focus,2002,231000.0,Farmington,False,4.0,True,False,False,55024,999.0
158318,INFINITI,I30,1997,320058.0,Spanaway,False,6.0,False,False,False,98387,999.0
166047,Mazda,MAZDA6,2003,175000.0,Manchester,False,6.0,False,False,False,03103,999.0
326052,Ford,Taurus,1997,104537.0,Edison,False,2.0,False,False,False,8817,999.0
481932,Mercury,Grand Marquis,1999,177832.0,Mchenry,False,4.0,True,False,False,60051,999.0
527526,Kia,Spectra,2004,NaN,Peninsula,False,4.0,False,False,False,44264,999.0


In [30]:
df.loc[df["price"] == 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
32211    44
55024     7
55906     3
7608      2
56301     2
57401     2
84054     2
6026      1
12553     1
55024     1
98387     1
03103     1
8817      1
60051     1
44264     1
60636     1
22191     1
20748     1
49058     1
37601     1
30458     1
32714     1
34748     1
50401     1
33619     1
33771     1
33168     1
56258     1
68701     1
41042     1
56387     1
84070     1
84095     1
59601     1
84047     1
72764     1
66720     1
68901     1
89048     1
91786     1
Name: count, dtype: int64

In [31]:
df.loc[
    df["price"] < 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
1320599,Buick,Century,1999,190000.0,Belle Glade,False,4.0,False,False,False,33430,165.0
1359718,Ford,Explorer,2005,150000.0,Medley,False,4.0,False,False,False,33178,200.0
878365,Buick,Century,2005,202158.0,Traverse City,False,5.0,True,False,False,49684,249.0
1271923,Acura,RL,2006,NaN,Bainbridge,False,3.0,False,False,False,39817,250.0
1271913,Mitsubishi,Lancer,2008,NaN,Bainbridge,False,2.0,True,False,False,39817,250.0
1271928,Nissan,Altima Coupe,2008,NaN,Bainbridge,False,5.0,True,False,False,39817,250.0
1271918,Toyota,Avalon,2005,NaN,Bainbridge,False,5.0,True,False,False,39817,250.0
1271941,Kia,Sorento,2006,NaN,Bainbridge,False,6.0,True,False,False,39817,250.0
1938898,Mercury,Villager,1998,104000.0,Henryetta,False,3.0,False,False,False,74437,256.0
141899,Chrysler,Town & Country,2007,NaN,Mount Morris,False,1.0,False,False,False,48458,299.0


In [32]:
df.loc[df["price"] < 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
29073    83
60914    31
34266    11
33430    10
33935    10
         ..
81082     1
89801     1
80204     1
80920     1
81003     1
Name: count, Length: 168, dtype: int64

CHECK POINT 1 COMPLETE
Target Price Checkpoint

- `price` has no missing, zero, or negative values.
- 494 listings are priced below $1,000.
- Repeated prices such as $999 appear across substantially different makes, models, years, mileage, and condition histories
- Many ultra-low prices are concentrated in specific dealer ZIP codes, suggesting placeholder or promotional pricing.

Current Conclusion

The repeated ultra-low prices across dissimilar vehicles, combined with their
concentration in specific dealer locations, suggest that some values below
$1,000 may represent dealer-specific pricing conventions, placeholder prices,
deposits, or other values that do not reflect full vehicle market value.

These records have not yet been removed from the original dataset. The current
proposed policy is to exclude listings priced below $1,000 from the dataset used
to train the initial current-value model while preserving them in the original
data.

In [33]:
valid_prices = df.loc[df['price'] >= 1000].copy()


In [34]:
valid_prices.shape

(2662757, 21)

In [35]:
valid_prices



,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


The major players
Price — the target we must protect from bad labels.
Vehicle identity — make, model, trim, year.
Lifecycle — is_new, mileage, owner count.
Condition — accidents, frame damage, salvage.
Market context — city, dealer ZIP, seller rating.

The central relationship is:

Vehicle identity sets the expected value range; lifecycle and condition adjust it; market context may shift the listing price.

Consistency checks

Instead of inventing arbitrary mileage cutoffs, we look for conflicts:

New vehicle + extremely high mileage
Used vehicle + zero mileage
New vehicle + several owners
Old vehicle + marked new
Price wildly inconsistent with similar vehicles
Repeated prices concentrated in one dealer location

In [36]:
valid_prices["price"].min()

np.float64(1000.0)

In [37]:
valid_prices.drop(columns=["vehicle_damage_category"], inplace=True)

In [38]:
valid_prices.drop_duplicates(inplace=True)

In [39]:
valid_prices.info()

<class 'pandas.DataFrame'>
Index: 2662757 entries, 0 to 2663250
Data columns (total 20 columns):
 #   Column            Dtype  
---  ------            -----  
 0   city              str    
 1   dealer_zip        object 
 2   engine_cylinders  str    
 3   engine_type       str    
 4   frame_damaged     object 
 5   fuel_type         str    
 6   has_accidents     object 
 7   horsepower        float64
 8   is_new            bool   
 9   make_name         str    
 10  mileage           float64
 11  model_name        str    
 12  owner_count       float64
 13  price             float64
 14  salvage           object 
 15  seller_rating     float64
 16  transmission      str    
 17  trim_name         str    
 18  wheel_system      str    
 19  year              int64  
dtypes: bool(1), float64(5), int64(1), object(4), str(9)
memory usage: 543.6+ MB


In [40]:
valid_prices["engine_type"].equals (valid_prices["engine_cylinders"])

True

In [41]:
valid_prices.drop(columns =["engine_cylinders"], inplace = True)

In [42]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


CHECKPOINT 2 Mileage validation

In [48]:
valid_prices.loc[
    valid_prices["mileage"] >= 1_000_000,
    [
        "mileage",
        "year",
        "make_name",
        "model_name",
        "price",
        "is_new",
        "city",
        "engine_type",
        "owner_count"
    ]
].sort_values("mileage")

,mileage,year,make_name,model_name,price,is_new,city,engine_type,owner_count
1853098,1111111.0,2020,Chevrolet,Equinox,23620.0,True,Boise,I4,NaN
1853005,1111111.0,2020,Chevrolet,Silverado 1500,39523.0,True,Boise,V8,NaN
1855598,1111111.0,2020,Cadillac,XT5,45970.0,True,Boise,I4,NaN
1853986,1111111.0,2020,Chevrolet,Silverado 1500,41420.0,True,Boise,V8,NaN
1180953,1225238.0,2020,Ford,F-150,45149.0,True,Belleview,V6,NaN
185063,4290461.0,2020,Chevrolet,Silverado 1500,50775.0,True,Nappanee,V8,NaN
2242507,99999988.0,2019,RAM,3500 Chassis,52610.0,True,Houston,I6 Diesel,NaN


In [55]:
valid_prices.loc[
    (
        (valid_prices["make_name"] == "Chevrolet") & (valid_prices["model_name"] == "Equinox")
    ) | (
        (valid_prices["make_name"] == "Chevrolet") & (valid_prices["model_name"] == "Silverado 1500")
    ),
    [ "make_name", "model_name", "mileage", "year", "price", "is_new", "city", "engine_type", "owner_count"]
].sort_values('mileage').head(50)

,make_name,model_name,mileage,year,price,is_new,city,engine_type,owner_count
1483414,Chevrolet,Equinox,0.0,2020,21801.0,True,Bowling Green,I4,NaN
1483379,Chevrolet,Equinox,0.0,2020,21372.0,True,Bowling Green,I4,NaN
2135402,Chevrolet,Silverado 1500,0.0,2020,27317.0,True,Georgetown,V8,NaN
2135367,Chevrolet,Silverado 1500,0.0,2020,48315.0,True,Weimar,V8,NaN
1769178,Chevrolet,Silverado 1500,0.0,2020,56400.0,True,Havre,V8,NaN
1769076,Chevrolet,Silverado 1500,0.0,2020,68035.0,True,Havre,V8,NaN
1768813,Chevrolet,Silverado 1500,0.0,2020,50995.0,True,Conrad,V8,NaN
1000866,Chevrolet,Silverado 1500,0.0,2020,39645.0,True,Lawrenceville,I6 Diesel,NaN
1000602,Chevrolet,Silverado 1500,0.0,2020,36205.0,True,Lawrenceville,V8,NaN
1000576,Chevrolet,Equinox,0.0,2020,31330.0,True,Lawrenceville,I4,NaN


In [56]:
valid_prices.loc[
    valid_prices["mileage"] >= 1_000_000,
    "mileage"
] = np.nan

In [58]:
(valid_prices["mileage"] < 0).sum()

np.int64(0)

In [82]:
valid_prices.loc[
    valid_prices["mileage"] >= 999_999,
    "mileage"
] = np.nan

In [83]:
valid_prices.nlargest(10, "mileage")[
    ["make_name", "model_name", "year", "mileage", "price", "is_new"]
]

,make_name,model_name,year,mileage,price,is_new
755125,Ford,F-250 Super Duty,2001,400000.0,5995.0,False
1186639,Ford,F-350 Super Duty,2004,399900.0,9800.0,False
2182925,Dodge,RAM 2500,2007,399578.0,9900.0,False
1435171,Lexus,ES 350,2008,399496.0,6259.0,False
1746143,Chevrolet,Silverado 2500HD,2006,398823.0,13950.0,False
51250,Lincoln,Town Car,2011,398654.0,4500.0,False
2172617,Dodge,RAM 3500,2005,398229.0,15990.0,False
836264,Dodge,RAM 3500,2004,397902.0,15118.0,False
2442168,Dodge,RAM 2500,2004,397896.0,10995.0,False
832156,GMC,Sierra 2500HD,2006,397662.0,11990.0,False


In [59]:
valid_prices.loc[
    valid_prices["mileage"] == 0,  "is_new"
].value_counts(dropna = False)

is_new
True     178969
False         3
Name: count, dtype: int64

In [67]:
valid_prices.loc[(valid_prices["mileage"] == 0) & (valid_prices["is_new"] == False)]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
1035593,Charleston,29414,I3,False,Gasoline,False,78.0,False,Mitsubishi,0.0,Mirage G4,2.0,16354.7,False,3.615385,M,ES FWD,FWD,2018
1446904,Lansing,66043,V6 Biodiesel,False,Biodiesel,False,260.0,False,RAM,0.0,1500,NaN,46166.0,False,5.000000,A,Big Horn Crew Cab 4WD,4WD,2020
1991260,Crosby,77532,V6,False,Gasoline,False,450.0,False,Ford,0.0,F-150,NaN,60609.0,False,3.800000,A,SVT Raptor SuperCrew 4WD,4WD,2020


In [69]:
#Zero mileage is retained for new vehicles, but zero mileage on used vehicles is treated as missing because it is not credible as an odometer reading.
valid_prices.loc[
    (valid_prices["mileage"] == 0) &
    (valid_prices["is_new"] == False),
    "mileage"
] = np.nan

valid_prices.loc[
    (valid_prices["mileage"] == 0)&
    (valid_prices["is_new"] == False)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [70]:
valid_prices["mileage"].max()

np.float64(999999.0)

In [71]:
valid_prices.loc[
    valid_prices["mileage"] == valid_prices["mileage"].max()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2116043,Kingman,86409,I4,NaN,Gasoline,NaN,161.0,True,Hyundai,999999.0,Elantra GT,NaN,22907.0,NaN,3.5,A,FWD,FWD,2020


In [72]:
valid_prices.loc[
    (valid_prices["mileage"] >= 100000)&
    (valid_prices["is_new"] == True)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
56105,Englewood,7631,I4,NaN,Gasoline,NaN,138.0,True,Buick,785778.0,Encore,NaN,28230.0,NaN,4.285714,A,Preferred AWD,AWD,2020
57037,Englewood,7631,V6,NaN,Gasoline,NaN,308.0,True,Chevrolet,115111.0,Blazer,NaN,39260.0,NaN,4.285714,A,1LT AWD,AWD,2020
58046,Englewood,7631,V6,NaN,Gasoline,NaN,310.0,True,Buick,381519.0,Enclave,NaN,45765.0,NaN,4.285714,A,Essence AWD,4WD,2020
68505,Woburn,1801,I4,NaN,Gasoline,NaN,170.0,True,Chevrolet,104261.0,Equinox,NaN,32340.0,NaN,4.714286,A,1.5T LT AWD,4WD,2020
203429,Portsmouth,3801,V6,NaN,Gasoline,NaN,271.0,True,Jeep,434718.0,Cherokee,NaN,36585.0,NaN,4.444444,A,Limited 4WD,4WD,2020
428139,Tunkhannock,18657,V8,NaN,Gasoline,NaN,420.0,True,Chevrolet,149304.0,Silverado 1500,NaN,55200.0,NaN,4.833333,A,LTZ Crew Cab 4WD,4WD,2020
451359,Bath,14810,I4,NaN,Gasoline,NaN,170.0,True,Chevrolet,514033.0,Equinox,NaN,27999.0,NaN,4.562500,A,2.0T LT AWD,4WD,2020
750379,Cary,27511,I4,NaN,Gasoline,NaN,170.0,True,Nissan,139592.0,Rogue,NaN,27965.0,NaN,4.320000,A,S FWD,FWD,2020
980582,Mc Donald,37353,I4,NaN,Gasoline,NaN,141.0,True,Nissan,118335.0,Rogue Sport,NaN,24454.0,NaN,4.882353,CVT,SV FWD,FWD,2020
1045927,Charleston,29407,I4,NaN,Gasoline,NaN,170.0,True,Nissan,166923.0,Rogue,NaN,33116.0,NaN,3.680000,CVT,SL FWD,FWD,2019


In [73]:
valid_prices.groupby("is_new")["mileage"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)[["min", "50%", "90%", "95%", "99%", "max"]]

,min,50%,90%,95%,99%,max
is_new,,,,,,
False,1.0,41168.0,127764.3,154689.15,208780.0,400000.0
True,0.0,6.0,36.0,287.00,4769.0,999999.0


In [77]:
valid_prices.loc[
    valid_prices["is_new"] == True
].nlargest(20, "mileage")[
    ["make_name", "model_name", "year", "mileage", "price", "owner_count", ]
]

,make_name,model_name,year,mileage,price,owner_count,is_new
2116043,Hyundai,Elantra GT,2020,999999.0,22907.0,NaN,True
56105,Buick,Encore,2020,785778.0,28230.0,NaN,True
2464582,Ford,F-150,2020,631835.0,51427.0,NaN,True
2454610,Jeep,Grand Cherokee,2020,610288.0,42763.0,NaN,True
1596466,Chevrolet,Silverado 3500HD Chassis,2020,590072.0,45288.0,NaN,True
451359,Chevrolet,Equinox,2020,514033.0,27999.0,NaN,True
203429,Jeep,Cherokee,2020,434718.0,36585.0,NaN,True
58046,Buick,Enclave,2020,381519.0,45765.0,NaN,True
2440543,Ford,EcoSport,2020,360012.0,21499.0,NaN,True
2144275,Chevrolet,Camaro,2020,351123.0,26155.0,NaN,True


In [78]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 10_000)).sum()

np.int64(1349)

In [79]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 50_000)).sum()

np.int64(115)

In [80]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 100_000)).sum()

np.int64(30)

In [81]:
valid_prices.loc[
    (valid_prices['is_new'] == True) &
    (valid_prices['mileage'] > 50000),
    'mileage'
] =np.nan

valid_prices.loc[
    (valid_prices["is_new"] == True)&
    (valid_prices["mileage"] > 50000),
    'mileage'
].sum()

np.float64(0.0)

CHECK POINT 2 COMPLETE 
Mileage validation

Extreme placeholder mileages were changed to NaN.

Zero mileage was retained for new vehicles.

Zero mileage on used vehicles was changed to NaN.

Mileage above 50,000 on new vehicles was changed to NaN.

Vehicle rows were preserved.

CHECK POINT 3---> Owner-count validation
Vehicles marked as new should normally have no previous owners recorded.

In [87]:
valid_prices.groupby("is_new")["owner_count"].describe()[
    ["count", "min", "50%", "max"]
]

,count,min,50%,max
is_new,,,,
False,1480648.0,1.0,1.0,19.0
True,965.0,1.0,1.0,2.0


In [88]:
valid_prices.groupby("is_new")["owner_count"].apply(
    lambda column: column.isna().sum()
)

is_new
False      46182
True     1134962
Name: owner_count, dtype: int64

In [100]:
valid_prices.loc[
    (valid_prices["is_new"] == True) &
    (valid_prices["owner_count"] == 2)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
22518,Warren,48093,V8,False,Gasoline,False,420.0,True,Cadillac,5.0,Escalade,2.0,86550.0,False,4.600000,A,Luxury 4WD,4WD,2019
151345,Upper Saddle River,7458,I4,False,Gasoline,False,280.0,True,Alfa Romeo,11531.0,Giulia,2.0,43385.0,False,4.625000,A,AWD,AWD,2018
261063,Newport,4953,I4,False,Gasoline,False,160.0,True,Ford,12.0,Focus,2.0,22805.0,False,NaN,A,SE Hatchback,FWD,2018
276016,North Brunswick,8902,I4,False,Gasoline,False,141.0,True,Nissan,18174.0,Rogue Sport,2.0,24999.0,False,4.485714,CVT,S AWD,AWD,2019
387620,Gurnee,60031,I4,False,Gasoline,True,173.0,True,Dodge,NaN,Journey,2.0,16995.0,False,3.920635,A,SXT FWD,FWD,2018
465929,Watertown,13601,I4,False,Gasoline,False,147.0,True,Kia,2.0,Forte,2.0,19370.0,False,5.000000,A,LX,FWD,2018
466305,Watertown,13601,V6,False,Gasoline,False,290.0,True,Kia,2.0,Sorento,2.0,42330.0,False,5.000000,A,EX V6 AWD,AWD,2019
549778,Schaumburg,60173,I4,False,Gasoline,False,141.0,True,Honda,8189.0,HR-V,2.0,26877.0,False,3.608696,CVT,EX-L AWD with Navigation,AWD,2018
732884,Newport News,23601,I4,False,Gasoline,False,245.0,True,Ford,11159.0,Escape,2.0,27495.0,False,4.125000,A,SE FWD,FWD,2018
806491,Boone,28607,V8,False,Gasoline,False,395.0,True,RAM,35895.0,1500,2.0,36186.0,False,4.482759,A,Laramie Quad Cab 4WD,4WD,2019


CHECK POINT 3 ---- NOT COMPLETE 
We need to determine whether is_new or owner_count is unreliable before cleaning either one.

CHECKPOINT 4----> Year Validation


In [107]:
valid_prices['year'].min()

np.int64(1915)

In [126]:
valid_prices.loc[
    valid_prices['year'] == 2021,
    [
        "make_name", "model_name", "price", "mileage", "is_new", "year"
    ]
   
].head(60)

,make_name,model_name,price,mileage,is_new,year
602,Jeep,Compass,26111.0,0.0,True,2021
610,Jeep,Compass,26329.0,0.0,True,2021
657,Jeep,Compass,27381.0,0.0,True,2021
668,Jeep,Compass,27593.0,0.0,True,2021
675,Jeep,Compass,28393.0,0.0,True,2021
719,Kia,Soul,19245.0,12.0,True,2021
731,Kia,Soul,19245.0,6.0,True,2021
743,Jeep,Compass,29151.0,0.0,True,2021
753,Kia,Soul,20905.0,12.0,True,2021
769,Kia,Soul,20905.0,16.0,True,2021


In [136]:
valid_prices.loc[
    valid_prices['year'] == 2021,
    [
        "make_name", "model_name", "price", "mileage", "is_new", "year"
    ]
].head(30)

,make_name,model_name,price,mileage,is_new,year
602,Jeep,Compass,26111.0,0.0,True,2021
610,Jeep,Compass,26329.0,0.0,True,2021
657,Jeep,Compass,27381.0,0.0,True,2021
668,Jeep,Compass,27593.0,0.0,True,2021
675,Jeep,Compass,28393.0,0.0,True,2021
719,Kia,Soul,19245.0,12.0,True,2021
731,Kia,Soul,19245.0,6.0,True,2021
743,Jeep,Compass,29151.0,0.0,True,2021
753,Kia,Soul,20905.0,12.0,True,2021
769,Kia,Soul,20905.0,16.0,True,2021


In [128]:
valid_prices.loc[
    valid_prices["is_new"] == True
].nlargest(20, "year")[
    ["make_name", "model_name", "year", "mileage", "price", "is_new"]
]

,make_name,model_name,year,mileage,price,is_new
602,Jeep,Compass,2021,0.0,26111.0,True
610,Jeep,Compass,2021,0.0,26329.0,True
657,Jeep,Compass,2021,0.0,27381.0,True
668,Jeep,Compass,2021,0.0,27593.0,True
675,Jeep,Compass,2021,0.0,28393.0,True
719,Kia,Soul,2021,12.0,19245.0,True
731,Kia,Soul,2021,6.0,19245.0,True
743,Jeep,Compass,2021,0.0,29151.0,True
753,Kia,Soul,2021,12.0,20905.0,True
769,Kia,Soul,2021,16.0,20905.0,True


In [137]:
valid_prices.loc[
    (valid_prices["year"] == 1915) &
    (valid_prices["is_new"] == True),
    [
        "make_name",
        "model_name",
        "price",
        "mileage",
        "is_new",
        "year",
        "owner_count"
    ]
].head(30)

,make_name,model_name,price,mileage,is_new,year,owner_count


CHECK POINT 4 ----> COMPLETE
Keep the full year range 1915–2021. No invalid years were found.

CHECK POINT 5 ----> HORSEPOWER

In [138]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


In [142]:
valid_prices['horsepower'].isna().sum()

np.int64(150520)

In [144]:
valid_prices.loc[valid_prices['horsepower'].isna()].head(30)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
50,Bronx,10466,NaN,False,NaN,False,NaN,False,Subaru,19801.0,Impreza,1.0,17300.0,False,2.800000,CVT,2.0i Touring Wagon AWD,NaN,2018
162,Bronx,10466,V6,False,Gasoline,False,NaN,False,Mercedes-Benz,41672.0,C-Class,2.0,12500.0,False,2.800000,A,NaN,NaN,2012
261,Bronx,10466,I4,False,Gasoline,False,NaN,False,Volkswagen,42154.0,GTI,1.0,24300.0,False,2.800000,Dual Clutch,NaN,NaN,2018
272,Bronx,10466,NaN,False,Electric,False,NaN,False,Kia,26155.0,Soul EV,1.0,13000.0,False,2.800000,A,FWD,FWD,2017
407,Woodbury,11797,V6,False,Gasoline,False,NaN,False,Porsche,62385.0,Panamera,2.0,26995.0,False,2.963636,A,NaN,NaN,2012
421,East Hartford,6108,V6,False,Gasoline,True,NaN,False,Ford,93707.0,F-150,1.0,17433.0,False,4.377778,A,NaN,NaN,2016
440,East Hartford,6108,I4,NaN,Gasoline,NaN,NaN,True,Jeep,2.0,Compass,NaN,22256.0,NaN,4.377778,A,NaN,NaN,2020
485,Bohemia,11716,I5,False,Gasoline,True,NaN,False,Volkswagen,45063.0,Golf,1.0,10646.0,False,3.647059,A,NaN,NaN,2013
498,Bronx,10466,V6,False,Gasoline,False,NaN,False,Lexus,31160.0,GS 350,1.0,30000.0,False,2.800000,A,NaN,NaN,2016
499,Bohemia,11716,V6,False,Gasoline,False,NaN,False,Mercedes-Benz,38885.0,E-Class,1.0,21561.0,False,3.647059,A,NaN,NaN,2014


In [145]:
valid_prices.loc[valid_prices['horsepower'] == 0]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [148]:
valid_prices.loc[valid_prices['horsepower'] < 0]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [153]:
valid_prices.loc[
    valid_prices["horsepower"] == valid_prices["horsepower"].max()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
810693,Charlotte,28203,W16,False,Gasoline,False,1001.0,False,Bugatti,8597.0,Veyron,9.0,1244996.0,False,4.6875,A,16.4 Coupe AWD,AWD,2008
2545871,Costa Mesa,92627,W16,False,Gasoline,False,1001.0,False,Bugatti,9791.0,Veyron,4.0,955000.0,False,5.0000,A,16.4 Coupe AWD,AWD,2008


In [150]:
valid_prices['horsepower'].min()

np.float64(55.0)

In [154]:
valid_prices.loc[
    valid_prices["horsepower"] == valid_prices["horsepower"].min()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
1633111,Brookville,45309,I3,False,Gasoline,False,55.0,False,Chevrolet,147375.0,Metro,1.0,1000.0,False,3.857143,M,Hatchback FWD,FWD,2000


CHECKPOINT ----> 5 COMPLETE


Horsepower validation conclusion

No zero or negative values

Minimum and maximum are plausible

Missing values exist, but their cause is unknown

Leave them as NaN until the missing-value phase

CHECK POINT 6 -----> Seller rating validation


In [155]:
valid_prices['seller_rating'].min()

np.float64(1.0)

In [156]:
valid_prices['seller_rating'].max()

np.float64(5.0)

In [158]:
valid_prices['seller_rating'].isna().sum()

np.int64(38290)

In [162]:
valid_prices.loc[valid_prices['seller_rating'].isna()].head(30)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
10,Guaynabo,969,I4,False,Gasoline,False,237.0,False,Alfa Romeo,301.0,4C,2.0,97579.0,False,NaN,A,Launch Edition Coupe RWD,RWD,2015
12,Guaynabo,969,I6,False,Gasoline,False,320.0,False,BMW,6903.0,3 Series,2.0,58995.0,False,NaN,A,340i xDrive Sedan AWD,AWD,2016
17385,Auburn Hills,48326,I4,NaN,Gasoline,NaN,120.0,True,Kia,34.0,Rio,NaN,16651.0,NaN,NaN,CVT,LX FWD,FWD,2020
17400,Auburn Hills,48326,V6 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,283.0,False,Dodge,38358.0,Grand Caravan,1.0,18877.0,False,NaN,A,SXT FWD,FWD,2019
17407,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,17.0,Forte,NaN,20204.0,NaN,NaN,CVT,LXS FWD,FWD,2020
17413,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,14.0,Soul,NaN,19888.0,NaN,NaN,CVT,LX FWD,FWD,2021
17419,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,12.0,Soul,NaN,19888.0,NaN,NaN,CVT,LX FWD,FWD,2021
17420,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,19.0,Forte,NaN,20186.0,NaN,NaN,CVT,LXS FWD,FWD,2020
17529,Auburn Hills,48326,I4,NaN,Gasoline,NaN,120.0,True,Kia,26.0,Rio,NaN,16651.0,NaN,NaN,CVT,LX FWD,FWD,2020
